---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 1.1 - Hello World

**Scenario:** You're an AI engineer for ACME Manufacturing. They are interested in creating a high quality MLflow assistant for developers and ML Engineers in particular. In this experiment we build the baseline agent.

**Objective:** Write a simple completions call and test the response. This ensures the API key variables are working and introduces the MLflow Server.

> **No API key?** You can read through every cell and understand the concepts. Cells that call the model are clearly marked — skip those and pick back up at the next markdown section.

## Google/Gemini API Free Tier - [AI Studio](https://aistudio.google.com)

| Model | Category | RPM | TPM | RPD |
|---|---|---|---|---|
| Gemini 3.1 Flash Lite (preview) | Text-out models | 15 | 250K | 500 |
| Gemini 3 Flash (preview) | Text-out models | 5 | 250K | 20 |
| Gemini 2.5 Flash | Text-out models | 5 | 250K | 20 |

## Google/Gemini API Paid Tier - [AI Studio](https://aistudio.google.com)

| Model | Category | RPM | TPM | RPD | $/M Input | $/M Output |
|---|---|---|---|---|---|---|
| Gemini 3.1 Flash Lite (preview) | Text-out models | 4k |  4M | 150k | 0.25 | 1.50 |
| Gemini 3 Flash (preview) | Text-out models | 1k | 2M | 10k | .50 | 3.00 |
| Gemini 2.5 Flash | Text-out models | 1k | 1M | 10k | .30 | 2.50 |
| Gemini 2.5 Flash Lite | Text-out models | 4k | 4M | Unl. | .10 | .40 |

---
## 0 — Environment Setup

1. Follow steps in the 'training/setup' folder
2. cmd/ctrl + shift + p
3. Python: Select Interpreter
4. Select your recently created environment

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()  # reads GEMINI_API_KEY from .env

if os.environ["GEMINI_API_KEY_FREE"]:
    print("Key exists")

### Configuring the OpenAI client to point at Gemini

We use the **OpenAI SDK** pointed at Google's OpenAI-compatible endpoint. This is an increasingly common pattern — the SDK is a familiar interface, and swapping providers is just a matter of changing `base_url` and the API key.

| Provider | `base_url` | Key env var |
|---|---|---|
| OpenAI (default) | *(omit)* | `OPENAI_API_KEY` |
| **Gemini** ✓ | `https://generativelanguage.googleapis.com/v1beta/openai/` | `GEMINI_API_KEY` |
| Azure OpenAI | `https://<resource>.openai.azure.com/` | `AZURE_OPENAI_API_KEY` |
| Ollama (local) | `http://localhost:11434/v1` | *(none required)* |

In [ ]:
openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [ ]:
MODEL = "gemini-2.5-flash-lite"

---
## 1 — Autologging and the Experiment

MLflow staples:

- `mlflow.set_tracking_uri()` - Sets MLFLOW_TRACKING_URI environment variable
- `mlflow.set_experiment()` — creates (or resumes) a named experiment bucket for all runs
- `mlflow.openai.autolog()` — patches the OpenAI SDK to emit a trace for every `chat.completions.create()` call, capturing inputs, outputs, token counts, and latency automatically

Once these are set, **every call in this notebook is captured with no extra code**.

In [ ]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if not tracking_uri:
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(os.getenv("EXPERIMENT_1_NAME","mlflow-agent-1"))
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:5000)")

---
## 2 — First Completions Call

Before we think about evaluation, structure, or scaffolding — let's just build the thing. A system prompt and a single completions call. This is the PoC.

In [ ]:
openai_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user",   "content": "How do I turn on auto logging in MLflow?"},
    ],
    temperature=0.1,
    max_completion_tokens=512,
)

In [ ]:
SYSTEM_PROMPT = """You are an MLflow assistant."""

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I turn on auto logging?"},
    ],
    temperature=0.1,
    max_completion_tokens=512,
)

print(response.choices[0].message.content)

---

# 3 - More on Logging

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GEMINI_API_KEY_FREE"]
)

response = llm.invoke("Hello, how are you?")
print(response.content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
mlflow.langchain.autolog()
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GEMINI_API_KEY_FREE"]
)

response = llm.invoke("Hello, how are you?")
print(response.content)

In [ ]:
# This will turn on logging for all libraries installed at import
mlflow.autolog()

# Disable logging
mlflow.autolog(disable=True)
mlflow.openai.autolog(disable=True)

# Customize logging
mlflow.autolog(
    log_model_signatures=False,
    extra_tags={"YOUR_TAG": "VALUE"},
)

### Push to PROD...right?

The completions call works fine so far.

> **You (AI Engineer):** "Let's start collecting feedback as part of a pilot with mlflow users and see what goes wrong."

> **Sam (Product):** "How do we *know* it's good? Do we have any metrics? What if it gives someone outdated functions or design patterns? What if the next model update makes it worse? Have you discussed anything with legal?"

You don't have solid answers yet, and that's completely fine. But it's time to start buidling the **evaluation** set.